In [43]:
import pandas as pd

In [77]:
CPchem = pd.read_csv("chemical_annotations.csv")
CPchem

,BROAD_ID,CPD_NAME,CPD_NAME_TYPE,CPD_SAMPLE_ID,DOS_LIBRARY,SOURCE_NAME,CHEMIST_NAME,VENDOR_CATALOG_ID,CPD_SMILES,USERCOMMENT
0,BRD-A56675431-001-04-0,altizide,INN,SA82748,NaN,Prestwick Chemical Inc.,NaN,Prestw-721,NS(=O)(=O)c1cc2c(NC(CSCC=C)NS2(=O)=O)cc1Cl,NaN
1,BRD-A51829654-001-01-4,"BRL-15,572",common,SA82481,NaN,Biomol International Inc.,NaN,AC-536,OC(CN1CCN(CC1)c1cccc(Cl)c1)C(c1ccccc1)c1ccccc1,NaN
2,BRD-K04046242-001-03-6,equilin,primary-common,SA82922,NaN,Prestwick Chemical Inc.,NaN,Prestw-850,C[C@]12CC[C@H]3C(=CCc4cc(O)ccc34)[C@@H]1CCC2=O,NaN
3,BRD-K16508793-001-01-8,diazepam,INN,SA59660,NaN,MicroSource Discovery Systems Inc.,NaN,1900003,CN1c2ccc(Cl)cc2C(=NCC1=O)c1ccccc1,NaN
4,BRD-K09397065-001-01-6,SR 57227A,to-be-curated,SA82504,NaN,Biomol International Inc.,NaN,AC-561,NC1CCN(CC1)c1cccc(Cl)n1,NaN
...,...,...,...,...,...,...,...,...,...,...
30611,BRD-K47092271-001-01-7,BRD-K47092271,BROAD_CPD_ID,SA799574,Benzofuran Library,Broad Institute of MIT and Harvard,NaN,NaN,COCC(=O)Nc1ccc2O[C@@H]3[C@@H](C[C@H](CC(=O)NCC...,NaN
30612,BRD-K30358777-001-01-0,BRD-K30358777,BROAD_CPD_ID,SA800605,Benzofuran Library,Broad Institute of MIT and Harvard,NaN,NaN,COc1ccc(CNC(=O)C[C@@H]2C[C@H]3[C@H](Oc4ccc(NC(...,NaN
30613,BRD-K32423836-001-01-9,BRD-K32423836,BROAD_CPD_ID,SA799806,Benzofuran Library,Broad Institute of MIT and Harvard,NaN,NaN,COCCNC(=O)C[C@H]1C[C@@H]2[C@@H](Oc3ccc(NC(=O)C...,NaN
30614,BRD-K28250273-001-01-2,BRD-K28250273,BROAD_CPD_ID,SA1482018,Azetidine Nitrile Full Library,Broad Institute of MIT and Harvard,NaN,NaN,OC[C@H]1[C@H]([C@H](C#N)N1C(=O)Nc1cccc(F)c1)c1...,NaN


In [64]:
# Function to convert SMILES to InChIKey
from rdkit import Chem
from rdkit.Chem import inchi
from pandarallel import pandarallel
pandarallel.initialize()

def smiles_to_inchikey(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles, sanitize=True)
        if mol is None:
            return 'CANNOT_CONVERT'
        # Optionally: kekulize or re-canonize
        Chem.SanitizeMol(mol)
        return inchi.MolToInchiKey(mol)
    except:
        return 'CANNOT_CONVERT'

moshkov= pd.read_csv("../../The_Moshkov_Files/data/assay_cp_data_PUMA.csv")
# Apply the function to the SMILES column

moshkov = moshkov.iloc[:, :271]

hof= pd.read_csv("../../The_Hofmarcher_Files/CP_count_Hofmarcher.csv")
hof = hof.iloc[:, :210]

ha = pd.read_csv("../../The_Ha_Files/CP_count_Ha.csv")
ha = ha.iloc[:, :202]

INFO: Pandarallel will run on 40 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [65]:
import pandas as pd
import pandas as pd

def add_active_hit_count(
    df,
    smiles_col,
    activity_value,
    assay_cols=None,
    hit_count_col="n_active_hits"
) -> pd.DataFrame:
    """
    Add a column indicating how many assays each compound is active in.

    Parameters
    ----------
    df : pd.DataFrame
        Input data containing a SMILES or compound ID column plus assay-result columns.
    smiles_col : str
        Name of the SMILES or identifier column.
    activity_value : int or float
        The value that denotes an active readout in the assay columns.
    assay_cols : list[str] or None, default None
        List of assay columns. If None, all columns except `smiles_col` are used.
    hit_count_col : str, default "n_active_hits"
        Name of the column to be added for hit counts.

    Returns
    -------
    pd.DataFrame
        DataFrame with an added column showing the number of active assay hits per compound.
    """
    # Determine which columns to treat as assays
    if assay_cols is None:
        assay_cols = df.columns.drop(smiles_col)

    # Count active assays per row
    hit_counts = (df[assay_cols] == activity_value).sum(axis=1)

    # Copy and add hit count column
    df_result = df.copy()
    df_result[hit_count_col] = hit_counts

    return df_result


In [66]:
moshkov

,smiles,1_1,10_19,10_20,100_277,100_278,104_291,105_355,105_369,105_361,...,88_252,88_254,89_255,9_14,9_17,9_18,90_256,91_257,92_258,93_259
0,CCOC(=O)c1ccc(NC(=S)N2CCSC2c2ccc(OC)cc2)cc1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,O=C(CC1NC(=O)NC1=O)Nc1cccc2ccccc12,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CC(Nc1nc(nc2ccccc12)N1CCCC1)c1ccccc1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CCCC(Oc1ccc(Br)cc1)c1nc2c3cc(OC)c(OC)cc3nc(S)n2n1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CC1=C(C(NC(=O)N1)c1ccc(F)cc1)C(=O)OCc1ccc2OCOc2c1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16165,COC(=O)[C@H]1[C@H](CO)[C@H]2Cn3c(=O)c(\C=C\C)c...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16166,C\C=C\c1ccc2n(C[C@H]3[C@H](CO)[C@H](N(C)[C@@H]...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16167,C\C=C\c1ccc2n(C[C@@H]3[C@@H](CO)[C@@H](N(C)[C@...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16168,C\C=C\c1ccc2n(C[C@H]3[C@H](CO)[C@@H](C(=O)N[C@...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [67]:
import numpy as np

moshkov['inchikey'] = moshkov['smiles'].parallel_apply(smiles_to_inchikey)

# moshkov["key14"] = moshkov["inchikey"].str[:14]
# hof["key14"] = hof["INCHIKEY"].str[:14]

hof.replace(-1, np.nan, inplace=True)

# ha["key14"] = ha["InChIKey"].str[:14]


moshkov = moshkov.rename(columns = {"inchikey":"InChIKey"})
hof = hof.rename(columns = {"INCHIKEY":"InChIKey"})

moshkov = moshkov.drop(columns = ["smiles"])

In [68]:
moshkov.InChIKey.value_counts()

JEFNGIWYLXETLU-UHFFFAOYSA-N    1
FKMAUGDURCCEJG-MMWGEVLESA-N    1
OEKKMSVCCRMVHL-SEXOINJZSA-N    1
XCSADXMQLTVTGS-ZNMIVQPWSA-N    1
CXNSNJNJSOSVDN-DFIYOIEZSA-N    1
                              ..
COXNDRJZODRCCW-UHFFFAOYSA-N    1
WSQGAZSPVHYFJA-RZQQEMMASA-N    1
HFXZOAOZNJYYEY-CXJARPSQSA-N    1
GRFOSLOREGQSPJ-WENCSYSZSA-N    1
CWELBPBUWGSKQY-WQNARRNSSA-N    1
Name: InChIKey, Length: 16170, dtype: int64

In [69]:
merged = pd.merge(moshkov, hof, on="InChIKey", how="outer")
merged

,1_1,10_19,10_20,100_277,100_278,104_291,105_355,105_369,105_361,105_378,...,200,201,202,203,204,205,206,207,208,209
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24601,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24602,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24603,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24604,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN


In [70]:
merged.InChIKey.value_counts()

JEFNGIWYLXETLU-UHFFFAOYSA-N    1
QPCDPRLGXJQWSH-MECTWPEFSA-N    1
ARAMEYDTNQOQNB-UHFFFAOYSA-N    1
AQWOKNZRXXHZFI-UHFFFAOYSA-N    1
AQQZWSKKBHAKCI-UHFFFAOYSA-N    1
                              ..
KTXYXNLRJUMWJA-ZRCGQRJVSA-N    1
WEZMNWQXUMSVMO-GSHUGGBRSA-N    1
CFNBOAHXJHVCOD-QZMQVMSPSA-N    1
CBOPBGRHGQIOTR-BXOOBUKZSA-N    1
ZZZPZDFBUDZIOU-UHFFFAOYSA-N    1
Name: InChIKey, Length: 24606, dtype: int64

In [71]:
# Assuming `df` is already loaded
merged = add_active_hit_count(merged, activity_value= 1.0, smiles_col="InChIKey",
                                    assay_cols = None,
                                   )

print(f"Kept {len(merged)} compounds.")

Kept 24606 compounds.


In [80]:
CPchem['inchikey'] = CPchem['CPD_SMILES'].parallel_apply(smiles_to_inchikey)
CPchem = CPchem.rename(columns={"inchikey": "InChIKey"})
CPchem = CPchem[["InChIKey", "BROAD_ID"]]
# CPchem["key14"] = CPchem["inchikey"].str[:14]
CPchem

,InChIKey,BROAD_ID
0,VGLGVJVUHYTIIU-UHFFFAOYSA-N,BRD-A56675431-001-04-0
1,QJHCTHPYUOXOGM-UHFFFAOYSA-N,BRD-A51829654-001-01-4
2,WKRLQDKEXYKHJB-HFTRVMKXSA-N,BRD-K04046242-001-03-6
3,AAOVKJBEBIDNHE-UHFFFAOYSA-N,BRD-K16508793-001-01-8
4,WPVVMKYQOMJPIN-UHFFFAOYSA-N,BRD-K09397065-001-01-6
...,...,...
30611,NDUFHUOGHUIKBH-OFYLTFBOSA-N,BRD-K47092271-001-01-7
30612,UPMRBEOCYVNRMT-OXPRRSNCSA-N,BRD-K30358777-001-01-0
30613,HRBYSLULMJVYNQ-TXDFXGTJSA-N,BRD-K32423836-001-01-9
30614,NDKYDEPQZPJKDZ-ULQDDVLXSA-N,BRD-K28250273-001-01-2


In [97]:
# Gather overlaps from each source
overlap_frames = []
for name, df in zip(["merged"], [merged]):
    ov = (
        df[df["InChIKey"].isin(CPchem["InChIKey"])]      # rows that match master keys
          .merge(CPchem[["InChIKey", "BROAD_ID"]],    # bring in BROAD_ID
                 on="InChIKey", how="left")
    )                        
    overlap_frames.append(ov)

# Combine into a single DataFrame
all_overlaps = pd.concat(overlap_frames, ignore_index=True)

print(f"Total overlaps across all sources: {len(all_overlaps)}")
all_overlaps = all_overlaps.drop_duplicates(subset=["BROAD_ID"], keep="first").reset_index(drop=True)

Total overlaps across all sources: 24715


In [98]:
all_overlaps = all_overlaps[["n_active_hits", "BROAD_ID", "InChIKey"]]
all_overlaps

,n_active_hits,BROAD_ID,InChIKey
0,2,BRD-A01717918-001-05-5,JEFNGIWYLXETLU-UHFFFAOYSA-N
1,1,BRD-A02214146-001-06-4,TYLHRPBDFWODNS-UHFFFAOYSA-N
2,8,BRD-A02481876-001-08-1,HKGJEZIGDHFJFL-UHFFFAOYSA-N
3,3,BRD-A02927672-001-06-3,TXQUPVHAFRNVNR-UHFFFAOYSA-N
4,12,BRD-A03764094-001-09-3,QCGHTRAZOATGMB-UHFFFAOYSA-N
...,...,...,...
24710,0,BRD-K33086245-001-05-9,ZZSBFYCJHLWUNQ-UHFFFAOYSA-N
24711,0,BRD-K78010432-001-04-1,ZZUFCTLCJUWOSV-UHFFFAOYSA-N
24712,2,BRD-K96799727-001-01-7,ZZUZYEMRHCMVTB-UHFFFAOYSA-N
24713,4,BRD-K47278471-003-04-0,ZZVUWRFHKOJYTH-UHFFFAOYSA-N


In [99]:
all_overlaps.to_csv("promicuous_compounds_with_assay_data.csv", index=False)

In [100]:
all_overlaps

,n_active_hits,BROAD_ID,InChIKey
0,2,BRD-A01717918-001-05-5,JEFNGIWYLXETLU-UHFFFAOYSA-N
1,1,BRD-A02214146-001-06-4,TYLHRPBDFWODNS-UHFFFAOYSA-N
2,8,BRD-A02481876-001-08-1,HKGJEZIGDHFJFL-UHFFFAOYSA-N
3,3,BRD-A02927672-001-06-3,TXQUPVHAFRNVNR-UHFFFAOYSA-N
4,12,BRD-A03764094-001-09-3,QCGHTRAZOATGMB-UHFFFAOYSA-N
...,...,...,...
24710,0,BRD-K33086245-001-05-9,ZZSBFYCJHLWUNQ-UHFFFAOYSA-N
24711,0,BRD-K78010432-001-04-1,ZZUFCTLCJUWOSV-UHFFFAOYSA-N
24712,2,BRD-K96799727-001-01-7,ZZUZYEMRHCMVTB-UHFFFAOYSA-N
24713,4,BRD-K47278471-003-04-0,ZZVUWRFHKOJYTH-UHFFFAOYSA-N


In [101]:
import matplotlib.pyplot as plt

all_overlaps.n_active_hits.value_counts()

0     11040
1      6472
2      3039
3      1571
4       891
5       553
6       324
7       218
8       168
9       107
10       86
11       47
12       35
14       26
13       18
15       18
16       14
17       10
22        7
18        6
32        4
27        4
19        4
24        3
25        3
29        3
28        3
77        3
21        3
33        2
26        2
41        2
30        2
70        2
39        2
76        2
43        1
37        1
23        1
57        1
52        1
75        1
40        1
20        1
80        1
38        1
65        1
61        1
31        1
48        1
45        1
60        1
86        1
36        1
35        1
47        1
68        1
Name: n_active_hits, dtype: int64